# Notebook 01 — Data Exploration
Explore CSIC 2012 HTTP dataset before feature engineering.

In [ ]:
# !pip install -r ../requirements_train.txt  # run once

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_RAW  = Path("../data/raw")
DATA_PROC = Path("../data/processed")
DATA_PROC.mkdir(parents=True, exist_ok=True)

## 1. Parse raw CSIC 2012 HTTP log files

In [ ]:
import re

def parse_http_log(filepath):
    records, current = [], {}
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith(("GET ", "POST ", "PUT ")):
            if current:
                records.append(current)
            parts = line.split(" ")
            current = {"method": parts[0], "url": parts[1] if len(parts) > 1 else "",
                       "headers": {}, "body": ""}
        elif ": " in line and current:
            k, _, v = line.partition(": ")
            current["headers"][k.lower()] = v
        elif line == "" and current.get("method") == "POST" and i + 1 < len(lines):
            body = lines[i + 1].strip()
            if body and not body.startswith(("GET ", "POST ", "PUT ")):
                current["body"] = body
                i += 1
        i += 1
    if current:
        records.append(current)
    return pd.DataFrame(records)

df_normal_train = parse_http_log(DATA_RAW / "normalTrafficTraining.txt")
df_normal_test  = parse_http_log(DATA_RAW / "normalTrafficTest.txt")
df_anomalous    = parse_http_log(DATA_RAW / "anomalousTrafficTest.txt")
print(f"Normal train: {len(df_normal_train)}, Normal test: {len(df_normal_test)}, Anomalous: {len(df_anomalous)}")

## 2. Label and assign attack sub-classes

In [ ]:
df_normal_train["label"] = "normal";  df_normal_train["label_id"] = 0
df_normal_test["label"]  = "normal";  df_normal_test["label_id"]  = 0
df_anomalous["label"]    = "attack";  df_anomalous["label_id"]    = 1

df_all = pd.concat([df_normal_train, df_normal_test, df_anomalous], ignore_index=True)

def assign_class(row):
    if row["label"] == "normal":
        return "normal", 0
    txt = (str(row["url"]) + " " + str(row["body"])).lower()
    if re.search(r"union.*select|or \d+=\d+|drop table|--.*select", txt):
        return "sqli", 1
    if re.search(r"<script|onerror|javascript:|alert\(", txt):
        return "xss", 2
    if re.search(r"\.\./|/etc/passwd|boot\.ini", txt):
        return "lfi", 3
    return "other_attack", 4

labels = df_all.apply(assign_class, axis=1, result_type="expand")
df_all["attack_class"]    = labels[0]
df_all["attack_class_id"] = labels[1]
print(df_all["attack_class"].value_counts())

## 3. Visualise distributions

In [ ]:
df_all["url_len"]  = df_all["url"].str.len().fillna(0)
df_all["body_len"] = df_all["body"].str.len().fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for label, grp in df_all.groupby("label"):
    axes[0].hist(grp["url_len"].clip(0, 500), bins=40, alpha=0.6, label=label)
axes[0].set_title("URL length"); axes[0].legend()

df_all["attack_class"].value_counts().plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Class distribution")

df_all.groupby("label")["url_len"].mean().plot(kind="bar", ax=axes[2], color="coral")
axes[2].set_title("Mean URL length by label")

plt.tight_layout()
plt.savefig("../data/processed/01_distributions.png", dpi=120)
plt.show()

## 4. Save parsed CSV

In [ ]:
df_all.to_csv(DATA_PROC / "csic_parsed.csv", index=False)
print(f"Saved {len(df_all)} rows → data/processed/csic_parsed.csv")